# Defining Prompts

**Guide for this lesson.** Do the exercise in the **starter** `cli-project/mcp_server.py`; the finished version is in `cli-project-complete/mcp_server.py`.

MCP **prompts** are pre-built, well-tested instruction templates the server hands to clients — so users get expert-quality results instead of ad-hoc prompts they write themselves. A prompt returns a **list of messages** the client can send straight to Claude.

Our server adds a **`format`** prompt: reformat a document into clean Markdown (using the `edit_document` tool). A user *could* type "convert report.pdf to markdown," but the server's tested prompt is more reliable.

## What to modify

**File:** `cli-project/mcp_server.py`

**1. Add the import.** ⚠️ The lesson text says `from mcp.server.fastmcp import base` — that's **wrong** and will `ImportError`. The correct path (as in the complete server) is:
```python
from mcp.server.fastmcp.prompts import base
```

**2. Add the prompt:**
```python
@mcp.prompt(
    name="format",
    description="Rewrites the contents of the document in Markdown format.",
)
def format_document(
    doc_id: str = Field(description="Id of the document to format"),
) -> list[base.Message]:
    prompt = f"""
    Your goal is to reformat a document to be written with markdown syntax.

    The id of the document you need to reformat is:
    <document_id>
    {doc_id}
    </document_id>

    Add in headers, bullet points, tables, etc as necessary. Feel free to add in extra text, but don't change the meaning of the report.
    Use the 'edit_document' tool to edit the document. After the document has been edited, respond with the final version of the doc. Don't explain your changes.
    """

    return [base.UserMessage(prompt)]
```

## How it works

- **`@mcp.prompt(name=..., description=...)`** registers a prompt; params (with `Field` descriptions) become the prompt's arguments, just like tools.
- The function **returns a list of messages** — here one `base.UserMessage`. You could return several (a system-ish setup + user turn, or few-shot examples) to shape the conversation.
- The template **interpolates `doc_id`** and instructs Claude to use the **`edit_document` tool** — so this prompt *composes* with the tools and resources we already defined. That's the point of prompts: encode your domain expertise once, wired to your server's own capabilities.

**Best practices:** keep prompts central to the server's purpose, write specific (not vague) instructions, test with varied inputs, and give clear descriptions.

## Verify — list the prompt and render its messages

Imports the **complete** server, lists its prompts (name/description/args), then *renders* the `format` prompt for a given `doc_id` to see the exact messages a client would send.

> Restart the kernel first if you imported a different folder's `mcp_server` earlier. Set `folder = "cli-project"` to check your own edits.

In [1]:
import sys
import os
from dotenv import find_dotenv

folder = "cli-project-complete"   # switch to "cli-project" to check your own edits
PROJECT = os.path.join(os.path.dirname(find_dotenv()), "01-building-with-the-claude-api", "07-mcp", folder)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import mcp_server

prompts = await mcp_server.mcp.list_prompts()
print("Prompts:")
for p in prompts:
    args = [a.name for a in (p.arguments or [])]
    print("  " + p.name + " - " + p.description + "  args=" + str(args))

Prompts:
  format - Rewrites the contents of the document in Markdown format.  args=['doc_id']


In [2]:
# Render the prompt for a specific document
result = await mcp_server.mcp.get_prompt("format", {"doc_id": "report.pdf"})

for m in result.messages:
    text = getattr(m.content, "text", m.content)
    print("[" + m.role + "]")
    print(text)

[user]

    Your goal is to reformat a document to be written with markdown syntax.

    The id of the document you need to reformat is:
    <document_id>
    report.pdf
    </document_id>

    Add in headers, bullet points, tables, etc as necessary. Feel free to add in extra text, but don't change the meaning of the report.
    Use the 'edit_document' tool to edit the document. After the document has been edited, respond with the final version of the doc. Don't explain your changes.
    


## Testing in the inspector (optional)

`mcp dev mcp_server.py` → **Prompts** section → select `format`, enter a `doc_id`, and it shows the generated messages. Same Node/token caveats as the server-inspector lesson.

## Where this leaves us

The **server** now offers a `format` prompt (alongside tools + resources — all three MCP primitives). But the **client** can't surface it yet: `mcp_client.py`'s `list_prompts` / `get_prompt` are still `TODO`.

Next: **Prompts in the client** — wire those two methods so the CLI can expose `format` as a slash-command (e.g. `/format report.pdf`).